# Particle tracking on an unstructured grid

This notebook runs **forward** particle tracking with the MODFLOW 6 **particle-tracking (PRT)** model on an **unstructured Voronoi grid** — a mesh of irregular polygonal cells rather than a regular row-and-column grid. You will track two particles across the domain and read off the **temperature** each one experiences along its path.

By the end you will be able to build a flow model on a Voronoi (**DISV**, discretization-by-vertices) grid, run a heat-transport model on the same grid, and drive a PRT model from the flow output while sampling the transport result along the tracks. It is also a preview of tomorrow's topic, heat transport with the groundwater energy (**GWE**) model. The scenario is adapted from [this MODFLOW 6 example problem](https://modflow6-examples.readthedocs.io/en/master/_notebooks/ex-gwe-prt.html).

The notebook builds **three** simulations in order:

1. a **flow** (**GWF**) model on the Voronoi grid,
2. a **heat-transport** (**GWE**) model that consumes the flow output, and
3. a **PRT** model that also consumes the flow output; its tracks are then colored by the GWE temperature field.

Start with the units. MODFLOW has no built-in unit system, so keep every input consistent — here **years** for time and **meters** for length.

In [ ]:
time_unit = "years"
length_unit = "meters"

## Phase 1 — Build and run the flow model

Load a pre-built Voronoi grid instead of defining rows and columns. Read the binary grid file (`.grb`) that MODFLOW 6 wrote for this mesh with `MfGrdFile`, and pull out its `modelgrid` — a FloPy grid object (`mg`) you will reuse for every model. The grid covers a 2000 × 1000 m rectangular domain.

In [ ]:
from pathlib import Path

from flopy.mf6.utils import MfGrdFile

grb = MfGrdFile(Path("../data/prt_voronoi/ex-gwe-prt-gwf.disv.grb"))
mg = grb.modelgrid

This model has constant-head (**CHD**) boundaries on the left, right, and bottom edges, plus three wells (two pumping, one injection). On a Voronoi grid you cannot address cells by row and column, so use FloPy's `GridIntersect` to find which cells a boundary line crosses: `.intersect()` returns the `cellids` a `LineString` passes through, and `mg.intersect(x, y)` returns the cell containing a `Point`.

In [ ]:
from flopy.utils import GridIntersect
from shapely.geometry import LineString, Point

xmin, xmax = 0.0, 2000.0
ymin, ymax = 0.0, 1000.0

gi = GridIntersect(mg)

cells_left = gi.intersect(
    LineString([(xmin, ymin), (xmin, ymax)]), geo_dataframe=False
)["cellids"]
cells_right = gi.intersect(
    LineString([(xmax, ymin), (xmax, ymax)]), geo_dataframe=False
)["cellids"]
cells_bottom = gi.intersect(
    LineString([(xmin, ymin), (xmax, ymin)]), geo_dataframe=False
)["cellids"]
cells_wel = [
    mg.intersect(p.x, p.y)
    for p in [Point(1200, 500), Point(700, 200), Point(1600, 700)]
]

Set up the workspaces: a base folder for the scenario and a nested `gwf` folder for the flow simulation, so each of the three models keeps its files separate.

In [ ]:
example_name = "prt_voronoi"
base_ws = Path("models") / example_name
gwf_name = f"{example_name}-gwf"
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

#### Build the flow simulation

Construct the whole flow model in one cell — you have seen these packages before, so this moves quickly. The one new piece is the grid: use `flopy.mf6.ModflowGwfdisv()` (discretization by vertices) and feed it the vertices and `cell2d` connectivity read from the grid file, rather than `nrow`/`ncol`.

The well (**WEL**) and constant-head (**CHD**) packages are list-based, and both carry a `TEMPERATURE` auxiliary value that the heat-transport model will read as its inflow temperature. On a DISV grid a cell ID is `(layer, cell2d-index)` — a 2-tuple, not the `(layer, row, column)` of a structured grid. For example a CHD entry here is

```python
# [cellid, head, aux_temperature] = [(layer, cell2d), head, temperature]
[(0, icpl), 2.0, 100.0 * yc / ymax]
```

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name, version="mf6", exe_name="mf6", sim_ws=gwf_ws
)
gwf_tdis = flopy.mf6.ModflowTdis(
    gwf_sim, time_units=time_unit, perioddata=[[1.0, 1, 1.0]]
)
gwf = flopy.mf6.ModflowGwf(gwf_sim, modelname=gwf_name, save_flows=True)
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    print_option="SUMMARY",
    complexity="complex",
    outer_dvclose=1.0e-8,
    inner_dvclose=1.0e-8,
)
disv = flopy.mf6.ModflowGwfdisv(
    gwf,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
gwf_ic = flopy.mf6.ModflowGwfic(gwf, strt=1.0)
gwf_sto = flopy.mf6.ModflowGwfsto(gwf, ss=0, sy=0, steady_state={0: True})
wells = [[0, c, -0.05, 80.0] for c in cells_wel]
wells[2][-2] *= -100  # convert third well to injection, Q = +5.0
gwf_wel = flopy.mf6.ModflowGwfwel(
    gwf,
    pname="WELL",
    auxiliary="TEMPERATURE",
    maxbound=len(wells),
    save_flows=True,
    stress_period_data={0: wells},
)
gwf_npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    xt3doptions=True,
    k=10.0,
    save_saturation=True,
    save_specific_discharge=True,
)

chdlist = []
seen = set()
for icpl in cells_left:
    yc = mg.cell2d[icpl][2]
    chdlist.append([(0, icpl), 2.0, 100.0 * yc / ymax])
    seen.add(int(icpl))
for icpl in cells_right:
    chdlist.append([(0, icpl), 1.0, 0.0])
    seen.add(int(icpl))
for icpl in cells_bottom:
    if int(icpl) not in seen:
        chdlist.append([(0, icpl), 1.8, 80.0])

gwf_chd = flopy.mf6.ModflowGwfchd(
    gwf,
    pname="CHD",
    auxiliary="TEMPERATURE",
    stress_period_data=chdlist,
)
gwf_oc = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{gwf_name}.cbc",
    head_filerecord=f"{gwf_name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

Write and run the flow simulation. The `assert` stops the notebook if MODFLOW 6 does not converge.

In [ ]:
gwf_sim.write_simulation(silent=False)
success, buff = gwf_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

## Phase 2 — Build and run the heat-transport (GWE) model

Now build a **groundwater energy (GWE)** model — MODFLOW 6's heat-transport model — on the same Voronoi grid. It reads the flow field you just computed and moves heat through it by advection and conduction. This course has not covered GWE yet and PRT is the focus, so this also runs in a single cell.

Note that the GWE model uses a **different time discretization** than the flow model: it runs for 10⁶ days across 1000 time steps with a slight multiplier, letting the temperature field approach steady state. The many labeled constants (thermal conductivities, heat capacities, densities) are the material properties heat transport needs; you do not need to memorize them.

In [ ]:
import os

gwe_name = f"{example_name}-gwe"
gwe_ws = base_ws / "gwe"
gwe_ws.mkdir(exist_ok=True, parents=True)

gwe_sim = flopy.mf6.MFSimulation(sim_name=gwe_name, sim_ws=gwe_ws, exe_name="mf6")
gwe_tdis = flopy.mf6.ModflowTdis(
    gwe_sim, time_units=time_unit, perioddata=[[1.0e6, 1000, 1.003]]
)
gwe = flopy.mf6.MFModel(
    gwe_sim,
    model_type="gwe6",
    modelname=gwe_name,
    model_nam_file=f"{gwe_name}.name",
)
nouter = 1000
ninner = 200
hclose = 1e-6
rclose = 1e-6
relax = 1.0
imsgwe = flopy.mf6.ModflowIms(
    gwe_sim,
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwe_name}.ims",
)
gwe_sim.register_ims_package(imsgwe, [gwe.name])
gwe_disv = flopy.mf6.ModflowGwedisv(
    gwe,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
strt_temp = 10.0  # initial temperature (°C)
alh = 0.0  # longitudinal mechanical dispersivity (m)
ath1 = 0.0  # transverse mechanical dispersivity (m)
ktw = 0.56 * 86400  # thermal conductivity of water, W/(m·°C) → J/(m·day·°C)
kts = 2.5 * 86400  # thermal conductivity of solids, W/(m·°C) → J/(m·day·°C)
rhow = 1000.0  # density of water (kg/m³)
cpw = 4180.0  # heat capacity of water (J/(kg·°C))
rhos = 2650.0  # density of dry solid (kg/m³)
cps = 900.0  # heat capacity of dry solid (J/(kg·°C))
lhv = 2500.0  # latent heat of vaporization (J/kg)
gwe_ic = flopy.mf6.ModflowGweic(gwe, strt=strt_temp)
gwe_adv = flopy.mf6.ModflowGweadv(gwe, scheme="TVD")
gwe_cnd = flopy.mf6.ModflowGwecnd(
    gwe,
    alh=alh,
    ath1=ath1,
    ktw=ktw,
    kts=kts,
)
porosity = 0.1
gwe_est = flopy.mf6.ModflowGweest(
    gwe,
    porosity=porosity,
    heat_capacity_water=cpw,
    density_water=rhow,
    latent_heat_vaporization=lhv,
    heat_capacity_solid=cps,
    density_solid=rhos,
)
gwe_ssm = flopy.mf6.ModflowGwessm(
    gwe,
    sources=[("WELL", "AUX", "TEMPERATURE"), ("CHD", "AUX", "TEMPERATURE")],
)
gwe_oc = flopy.mf6.ModflowGweoc(
    gwe,
    budget_filerecord=f"{gwe_name}.cbc",
    temperature_filerecord=f"{gwe_name}.ucn",
    saverecord={0: [("TEMPERATURE", "ALL"), ("BUDGET", "ALL")]},
    printrecord=[("TEMPERATURE", "LAST"), ("BUDGET", "LAST")],
)
rel_gwf_ws = os.path.relpath(gwf_ws, gwe_ws)
gwe_fmi = flopy.mf6.ModflowGwefmi(
    gwe,
    packagedata=[
        ("GWFHEAD", Path(f"{rel_gwf_ws}/{gwf_name}.hds"), None),
        ("GWFBUDGET", Path(f"{rel_gwf_ws}/{gwf_name}.cbc"), None),
    ],
)

Write and run the GWE simulation. It reads the flow model's head and budget files through its own flow model interface (**FMI**) package, configured at the bottom of the cell above.

In [ ]:
gwe_sim.write_simulation(silent=False)
success, buff = gwe_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

## Phase 3 — Track particles with PRT

Now to the main event: the **particle-tracking (PRT)** model. Set it up like the other MODFLOW 6 models — simulation, TDIS, the PRT model with a matching **DISV** grid, and the Model Input Parameters (**MIP**) package that sets porosity.

In [ ]:
prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name, version="mf6", exe_name="mf6", sim_ws=prt_ws
)
prt_tdis = flopy.mf6.ModflowTdis(
    prt_sim, time_units=time_unit, perioddata=[[1.0, 1, 1.0]]
)
prt = flopy.mf6.ModflowPrt(prt_sim, modelname=prt_name)
prt_disv = flopy.mf6.ModflowPrtdisv(
    prt,
    nlay=mg.nlay,
    ncpl=mg.ncpl,
    nvert=mg.nvert,
    vertices=mg._vertices,
    cell2d=mg.cell2d,
    top=mg.top[0],
    botm=mg.top_botm[1][0],
)
prt_mip = flopy.mf6.ModflowPrtmip(prt, pname="mip", porosity=porosity)

#### Define the release points

This scenario was adapted from a PRT test case that releases many evenly spaced particles along the left edge; here you keep just two of them. Use `mg.intersect(x, y)` to find the cell each release point falls in. Unlike MP7, PRT wants release points in **model coordinates** (not cell-local coordinates), and the point must lie inside the cell you name — so each release record supplies both the cell ID and the x, y, z coordinates. (Supplying both is redundant and may change in future.)

Add the particle release point (**PRP**) package with `flopy.mf6.ModflowPrtprp()`, passing the two release records as `packagedata` and releasing them at the start of the run (`perioddata={0: ["FIRST"]}`). Set `extend_tracking=True` so PRT keeps following particles past the simulation's end time until they terminate naturally, rather than stopping them mid-path.

In [ ]:
rpts = [[20, i, 0.5] for i in range(1, 999, 20)]
prpdata = [
    (i, (0, mg.intersect(p[0], p[1])), p[0], p[1], p[2])
    for i, p in enumerate([rpts[23], rpts[32]])
]

prt_prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(prpdata),
    packagedata=prpdata,
    perioddata={0: ["FIRST"]},
    extend_tracking=True,
)

Add the remaining PRT packages. Configure output control (**OC**) to write tracks to CSV, and connect the model to the flow results with the flow model interface (**FMI**) package, pointing it at the flow model's head and budget files.

In [ ]:
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    trackcsv_filerecord=[f"{prt_name}.trk.csv"],
)
prt_fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws}/{gwf_name}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws}/{gwf_name}.cbc"),
    ],
)

Finally add the explicit model solution (**EMS**) with `flopy.mf6.ModflowEms()`. Unlike flow and transport, PRT does not use an iterative matrix solver — it advances particles directly — so it needs an EMS, not an IMS. Giving a PRT model an IMS instead makes MODFLOW 6 raise an error.

In [ ]:
prt_ems = flopy.mf6.ModflowEms(prt_sim, pname="ems")
prt_sim.register_solution_package(prt_ems, [prt.name])

Write and run the particle-tracking simulation.

In [ ]:
prt_sim.write_simulation(silent=False)
success, buff = prt_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

## Combine the results: temperature along the tracks

Bring the three models together. Load the GWE temperature field with `gwe.output.temperature()` and the PRT tracks from the CSV with `pd.read_csv()`. Then sample the temperature at every point along each track: build a `CloughTocher2DInterpolator` — a smooth 2D interpolator — from the Voronoi cell centers and their temperatures, and evaluate it at the particle x, y positions, storing the result in a new `therm` column. Finally split the pathlines into the two particles by their release-point index (`irpt`).

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator

temperatures = gwe.output.temperature().get_data()[0, 0]
pathlines = pd.read_csv(prt_ws / f"{prt_name}.trk.csv", na_filter=False)

xc = mg.xcellcenters
yc = mg.ycellcenters
interp = CloughTocher2DInterpolator(list(zip(xc, yc, strict=False)), temperatures)
pathlines["therm"] = interp(pathlines["x"].values, pathlines["y"].values)

particle_a = pathlines[pathlines["irpt"] == 1]
particle_b = pathlines[pathlines["irpt"] == 2]

#### Plot the results

Make a three-panel figure. The top map view shows the temperature field, specific-discharge vectors (`.plot_vector()`), the wells, and the two particle pathlines. The lower two panels plot the temperature each particle sees versus its x position and versus its travel time, using colored line segments so temperature is encoded along the path. Specific discharge is recovered from the flow budget with `get_specific_discharge()`.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

spdis = gwf.output.budget().get_data(text="DATA-SPDIS")[0]
qx, qy, _ = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(6, 7),
    tight_layout=True,
    gridspec_kw={"height_ratios": [3, 1, 1]},
)

# map view
ax = axes[0]
ax.set_xlim(0, 2000)
ax.set_ylim(0, 1000)
ax.set_aspect("equal")
pmv = flopy.plot.PlotMapView(model=gwf, ax=ax)
pmv.plot_grid(alpha=0.25)
tempmesh = pmv.plot_array(temperatures, alpha=0.7)
cv = pmv.contour_array(temperatures, levels=np.linspace(0, 80, 9))
plt.clabel(cv, colors="k")
plt.colorbar(
    tempmesh,
    shrink=0.5,
    ax=ax,
    label="Temperature (°C)",
    location="bottom",
    fraction=0.1,
)
pmv.plot_vector(qx, qy, normalize=True, alpha=0.25)
pmv.plot_bc(ftype="WELL")
for ipl, (_, pl) in enumerate(pathlines.groupby(["iprp", "irpt", "trelease"])):
    pl.plot(
        kind="line", linestyle="--", x="x", y="y", ax=ax, legend=False, color="blue"
    )
    if ipl == 0:
        ax.annotate(
            "Particle 1",
            xy=(1050, 380),
            xycoords="data",
            xytext=(30, -20),
            textcoords="offset points",
            bbox={"boxstyle": "round", "fc": "1.0", "alpha": 0.66},
            arrowprops={
                "arrowstyle": "->",
                "shrinkA": 0,
                "shrinkB": 5,
                "connectionstyle": "angle,angleA=0,angleB=135,rad=40",
            },
        )
    else:
        ax.annotate(
            "Particle 2",
            xy=(1050, 610),
            xycoords="data",
            xytext=(-75, 10),
            textcoords="offset points",
            bbox={"boxstyle": "round", "fc": "1.0", "alpha": 0.66},
            arrowprops={
                "arrowstyle": "->",
                "shrinkA": 0,
                "shrinkB": 5,
                "connectionstyle": "angle,angleA=0,angleB=135,rad=30",
            },
        )
ax.legend(
    handles=[
        mpl.lines.Line2D(
            [0],
            [0],
            marker=">",
            linestyle="",
            color="grey",
            markerfacecolor="gray",
            label="Specific discharge",
        ),
        mpl.lines.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markerfacecolor="red",
            label="Well",
        ),
    ],
    loc="lower right",
)

# shared colour norm for the two profile plots
norm = plt.Normalize(
    min(particle_a["therm"].min(), particle_b["therm"].min()) - 5,
    max(particle_a["therm"].max(), particle_b["therm"].max()) + 5,
)

# temperature vs x-position
for part in [particle_a, particle_b]:
    pts = np.array([part["x"], part["therm"]]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, norm=norm)
    lc.set_array(part["therm"])
    lc.set_linewidth(3)
    axes[1].add_collection(lc)
axes[1].annotate("Particle 2", xy=(400, 68), xycoords="data")
axes[1].annotate("Particle 1", xy=(400, 50), xycoords="data")
axes[1].set_xlabel("X (m)")
axes[1].set_xlim(0, 2000)
axes[1].set_xticks(np.arange(0, 2100, 250))
axes[1].set_ylabel("Temperature (°C)")
axes[1].set_ylim(40, 80)
axes[1].set_yticks(np.arange(40, 81, 10))

# temperature vs travel time
for part in [particle_a, particle_b]:
    pts = np.array([part["t"], part["therm"]]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc = LineCollection(segs, norm=norm)
    lc.set_array(part["therm"])
    lc.set_linewidth(3)
    axes[2].add_collection(lc)
axes[2].annotate("Particle 2", xy=(15000, 68), xycoords="data")
axes[2].annotate("Particle 1", xy=(15000, 50), xycoords="data")
axes[2].set_xlabel("Time (days)")
axes[2].set_xlim(0, max(particle_a["t"].max(), particle_b["t"].max()))
axes[2].set_ylabel("Temperature (°C)")
axes[2].set_ylim(40, 80)
axes[2].set_yticks(np.arange(40, 81, 10))

plt.show()

**What to look for.** In the top map the dashed blue lines are the two particle paths; they follow the discharge vectors, bending around the pumping and injection wells and tracing the flow from the left boundary toward the right. The color contours are the steady-state temperature field. In the lower panels the line color is the temperature each particle experiences: as a particle moves through warmer and cooler zones its curve rises and falls, and plotting the same temperatures against travel time (bottom) rather than distance (middle) shows how long the particle lingers in each thermal zone — a particle that slows down spends more time at a given temperature even though it covers little distance.

## Recap

In this notebook you:

- loaded a pre-built **unstructured Voronoi grid** and used `GridIntersect` to place boundaries and release points by geometry instead of by row and column;
- built three MODFLOW 6 models on that one grid — a **flow (GWF)** model, a **heat-transport (GWE)** model, and a **particle-tracking (PRT)** model — each on a **DISV** discretization, with GWE and PRT both reading the flow output through their flow model interface (**FMI**) packages;
- ran **forward** tracking with `extend_tracking` so particles were followed to their natural ends; and
- combined the transport and tracking results by interpolating the GWE temperature field onto the PRT paths, revealing the temperature history each particle experiences in space and in time.